# Smoke Tests: RSPrompter & DSMPrompter

Ce notebook vérifie que les deux pipelines sont correctement configurés **avant** de lancer un entraînement complet.

**Ce qui est testé :**
- Imports et dépendances
- Checkpoints présents sur le disque
- Initialisation des modèles (sans données réelles)
- Forward pass avec des données dummy (forme des sorties)
- Calcul de la loss

**Ce qui N'est PAS requis :** GPU (CPU suffit pour les checks), données réelles.

---
**Lancer depuis la racine du projet** : `jupyter notebook notebooks/smoke_test_rsprompter_dsmprompter.ipynb`

## 0. Setup — Paths & Imports

In [ ]:
import os
import sys

# Ensure we run from project root so relative imports work
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU — forward pass tests will use CPU")

In [ ]:
# Check key dependencies
deps = {
    "segment_anything": "SAM (ViT-H) — needed by RSPrompter",
    "sam2": "SAM2 — needed by DSMPrompter",
    "detectron2": "Detectron2 — needed by Mask R-CNN / Faster R-CNN",
    "pycocotools": "COCO tools",
    "scipy": "Hungarian matcher (linear_sum_assignment)",
    "cv2": "OpenCV",
}

all_ok = True
for pkg, desc in deps.items():
    try:
        __import__(pkg)
        print(f"  OK  {pkg:25s} — {desc}")
    except ImportError as e:
        print(f"  MISSING  {pkg:25s} — {desc}  ({e})")
        all_ok = False

print()
if all_ok:
    print("All dependencies found.")
else:
    print("Some dependencies are missing — install them before training.")

In [ ]:
# Checkpoint paths (mirroring the constants in the model files)
BASE_MODELS = os.path.join(PROJECT_ROOT, "src", "models")
DATA_ROOT   = os.path.abspath(os.path.join(BASE_MODELS, "../../../../Output"))

SAM_CHECKPOINT  = os.path.abspath(os.path.join(BASE_MODELS, "../../../../sam_vit_h_4b8939.pth"))
SAM2_CHECKPOINT = os.path.abspath(os.path.join(BASE_MODELS, "../../../../checkpoints/sam2.1_hiera_large.pt"))
EMBED_DIR       = os.path.join(BASE_MODELS, "sam_embeddings_vith")
TRAIN_JSON      = os.path.join(DATA_ROOT, "coco", "train.json")
VAL_JSON        = os.path.join(DATA_ROOT, "coco", "val.json")
TEST_JSON       = os.path.join(DATA_ROOT, "coco", "test.json")

print("--- Checkpoint / Data Paths ---")
paths = {
    "SAM ViT-H checkpoint":         SAM_CHECKPOINT,
    "SAM2 large checkpoint":         SAM2_CHECKPOINT,
    "SAM embeddings dir (RSPrompter step 1)": EMBED_DIR,
    "train.json":                    TRAIN_JSON,
    "val.json":                      VAL_JSON,
    "test.json":                     TEST_JSON,
}
for label, path in paths.items():
    exists = os.path.exists(path)
    status = "OK  " if exists else "MISS"
    print(f"  [{status}]  {label:45s}  {path}")

---
## 1. RSPrompter Pipeline

**Architecture** : `EmbeddingDataset` (charge des `.pt` SAM pré-calculés) → `RSPrompterFast` (SAM decoder + `PromptEncoder` entraînable) → `RSPrompterLoss` (Hungarian matching)

**Prérequis** :
1. `sam_vit_h_4b8939.pth` présent
2. Embeddings pré-calculés via `src/misc/Pre_Compute_SAM.py` (step 1)

In [ ]:
# --- Test 1.1: Imports ---
print("Test 1.1 — RSPrompter imports...")

sys.path.insert(0, BASE_MODELS)
try:
    from RSPrompter import (
        EmbeddingDataset, RSPrompterFast, RSPrompterLoss,
        HungarianMatcher, PromptEncoder, collate_fn
    )
    print("  PASS  All RSPrompter classes imported successfully")
except ImportError as e:
    print(f"  FAIL  ImportError: {e}")

In [ ]:
# --- Test 1.2: EmbeddingDataset with dummy .pt files ---
print("Test 1.2 — EmbeddingDataset with dummy embeddings...")

import tempfile, json
import torch
import numpy as np

# Create a tiny fake COCO JSON + fake .pt embedding files
with tempfile.TemporaryDirectory() as tmpdir:
    embed_dir = os.path.join(tmpdir, "embeddings")
    os.makedirs(embed_dir)

    # Fake COCO JSON (2 images, 3 annotations)
    fake_coco = {
        "images": [
            {"id": 1, "file_name": "img_001.png", "height": 1024, "width": 1024},
            {"id": 2, "file_name": "img_002.png", "height": 1024, "width": 1024},
        ],
        "annotations": [
            {"id": 1, "image_id": 1, "category_id": 1, "bbox": [100, 100, 200, 150], "segmentation": []},
            {"id": 2, "image_id": 1, "category_id": 2, "bbox": [300, 300, 100, 100], "segmentation": []},
            {"id": 3, "image_id": 2, "category_id": 1, "bbox": [50, 50, 300, 200],  "segmentation": []},
        ],
        "categories": [{"id": i, "name": f"tree_{i}"} for i in range(1, 8)]
    }
    json_path = os.path.join(tmpdir, "train.json")
    with open(json_path, "w") as f:
        json.dump(fake_coco, f)

    # Fake SAM embeddings: (256, 64, 64) per image
    for img_id in [1, 2]:
        emb = torch.randn(256, 64, 64, dtype=torch.float32)
        torch.save(emb, os.path.join(embed_dir, f"{img_id}.pt"))

    ds = EmbeddingDataset(json_path, embed_dir)
    assert len(ds) == 2, f"Expected 2 samples, got {len(ds)}"

    sample = ds[0]
    emb = sample["embedding"]
    assert emb.shape == (256, 64, 64), f"Embedding shape {emb.shape} != (256,64,64)"
    assert "targets" in sample
    assert "boxes" in sample["targets"] and "masks" in sample["targets"]
    print(f"  PASS  Dataset len={len(ds)}, embedding shape={tuple(emb.shape)}, boxes={sample['targets']['boxes'].shape}")

In [ ]:
# --- Test 1.3: PromptEncoder forward pass (no SAM checkpoint needed) ---
print("Test 1.3 — PromptEncoder standalone forward pass...")

enc = PromptEncoder(embed_dim=256)
dummy_emb = torch.randn(2, 256, 64, 64)  # batch=2
box_out, score_out = enc(dummy_emb)
assert box_out.shape   == (2, 4, 64, 64), f"box_out shape: {box_out.shape}"
assert score_out.shape == (2, 1, 64, 64), f"score_out shape: {score_out.shape}"
print(f"  PASS  box_out={tuple(box_out.shape)}, score_out={tuple(score_out.shape)}")

In [ ]:
# --- Test 1.4: HungarianMatcher ---
print("Test 1.4 — HungarianMatcher...")

matcher = HungarianMatcher()
B, N = 2, 10  # batch=2, 10 proposals

fake_outputs = {
    "pred_logits": torch.randn(B, N, 1),
    "pred_boxes":  torch.rand(B, N, 4) * 1024,
    "pred_masks":  torch.randn(B, N, 1, 256, 256),
}
fake_targets = [
    {"boxes": torch.rand(3, 4) * 1024, "masks": torch.zeros(3, 1024, 1024)},
    {"boxes": torch.rand(2, 4) * 1024, "masks": torch.zeros(2, 1024, 1024)},
]

indices = matcher(fake_outputs, fake_targets)
assert len(indices) == B
src0, tgt0 = indices[0]
assert len(src0) <= 3, f"Matched {len(src0)} proposals to 3 GT — expected ≤3"
print(f"  PASS  batch[0]: {len(src0)} matched pairs, batch[1]: {len(indices[1][0])} matched pairs")

In [ ]:
# --- Test 1.5: RSPrompterLoss ---
print("Test 1.5 — RSPrompterLoss computation...")

criterion = RSPrompterLoss(HungarianMatcher())
loss = criterion(fake_outputs, fake_targets)
assert loss.item() == loss.item(), "Loss is NaN!"
assert loss.item() > 0
print(f"  PASS  loss = {loss.item():.4f}  (scalar, > 0)")

In [ ]:
# --- Test 1.6: RSPrompterFast full forward pass (requires SAM checkpoint) ---
print("Test 1.6 — RSPrompterFast full forward pass...")

if not os.path.exists(SAM_CHECKPOINT):
    print(f"  SKIP  SAM checkpoint not found: {SAM_CHECKPOINT}")
    print("         Download sam_vit_h_4b8939.pth and place it at the path above to run this test.")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Loading RSPrompterFast on {device} (this may take ~30s)...")
    model = RSPrompterFast(SAM_CHECKPOINT, num_proposals=5).to(device)
    model.eval()

    # Count trainable parameters
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    with torch.no_grad():
        dummy_emb = torch.randn(1, 256, 64, 64, device=device)  # batch=1, SAM embedding shape
        masks, ious, boxes, scores = model(dummy_emb)

    assert masks.shape  == (1, 5, 1, 256, 256), f"masks shape: {masks.shape}"
    assert ious.shape   == (1, 5, 1),            f"ious shape: {ious.shape}"
    assert boxes.shape  == (1, 5, 4),            f"boxes shape: {boxes.shape}"
    assert scores.shape == (1, 5),               f"scores shape: {scores.shape}"
    print(f"  PASS  masks={tuple(masks.shape)}, ious={tuple(ious.shape)}, boxes={tuple(boxes.shape)}")

    # Test loss end-to-end
    targets_1 = [{"boxes": torch.rand(3, 4, device=device)*1024, "masks": torch.zeros(3, 1024, 1024, device=device)}]
    outputs_1 = {"pred_logits": scores.unsqueeze(-1), "pred_boxes": boxes, "pred_masks": masks}
    criterion_gpu = RSPrompterLoss(HungarianMatcher())
    loss_e2e = criterion_gpu(outputs_1, targets_1)
    print(f"  PASS  End-to-end loss = {loss_e2e.item():.4f}")

In [ ]:
# --- Test 1.7: Pre_Compute_SAM dependencies check ---
print("Test 1.7 — Pre_Compute_SAM.py prerequisites...")

pre_compute_script = os.path.join(PROJECT_ROOT, "src", "misc", "Pre_Compute_SAM.py")
print(f"  Script: {'OK' if os.path.exists(pre_compute_script) else 'MISSING'}  {pre_compute_script}")
print(f"  SAM checkpoint: {'OK' if os.path.exists(SAM_CHECKPOINT) else 'MISSING'}")
print(f"  train images dir: {'OK' if os.path.isdir(os.path.join(DATA_ROOT, 'train', 'RGB')) else 'MISSING'}")
print(f"  val images dir:   {'OK' if os.path.isdir(os.path.join(DATA_ROOT, 'val', 'RGB')) else 'MISSING'}")

embed_train = os.path.join(EMBED_DIR, "train")
embed_val   = os.path.join(EMBED_DIR, "val")
if os.path.isdir(embed_train):
    n_train = len([f for f in os.listdir(embed_train) if f.endswith(".pt")])
    n_val   = len([f for f in os.listdir(embed_val) if f.endswith(".pt")]) if os.path.isdir(embed_val) else 0
    print(f"  Embeddings already computed: {n_train} train, {n_val} val")
else:
    print(f"  Embeddings NOT computed yet — run Pre_Compute_SAM.py first")
    print(f"    Command: python src/misc/Pre_Compute_SAM.py")

---
## 2. DSMPrompter Pipeline

**Architecture** : `PlantationsDataset` (charge RGB + DSM TIFF) → `DSMPrompterSAM2` (SAM2 frozen + `DSMEncoder` entraînable + `PromptProposalHead`) → `DSMLoss` (Hungarian matching)

**Prérequis** :
1. `checkpoints/sam2.1_hiera_large.pt` présent
2. Images DSM `.tif` dans `Output/train/DSM/` et `Output/val/DSM/`

**Note** : SAM2 utilise un chemin de config relatif (`configs/sam2.1/sam2.1_hiera_l.yaml`). Il faut lancer depuis la racine du projet **ou** depuis le dossier SAM2 installé.

In [ ]:
# --- Test 2.1: Imports ---
print("Test 2.1 — DSMPrompter imports...")

try:
    from DSM_Prompter import (
        DSMEncoder, PromptProposalHead, HungarianMatcher as DSMHungarianMatcher,
        DSMLoss, DSMPrompterSAM2, PlantationsDataset
    )
    print("  PASS  All DSMPrompter classes imported successfully")
except ImportError as e:
    print(f"  FAIL  ImportError: {e}")
    print("  Hint: Install SAM2 with: pip install git+https://github.com/facebookresearch/sam2.git")

In [ ]:
# --- Test 2.2: DSMEncoder standalone (no SAM2 needed) ---
print("Test 2.2 — DSMEncoder forward pass...")

enc_dsm = DSMEncoder(embed_dim=256)
# DSM input: (B, 1, 1024, 1024)
dummy_dsm = torch.randn(2, 1, 1024, 1024)
dsm_out = enc_dsm(dummy_dsm)
assert dsm_out.shape == (2, 256, 64, 64), f"DSMEncoder output shape: {dsm_out.shape}"
print(f"  PASS  Input (2, 1, 1024, 1024) → Output {tuple(dsm_out.shape)}")

# Check trainable params
trainable_dsm = sum(p.numel() for p in enc_dsm.parameters())
print(f"  DSMEncoder params: {trainable_dsm:,}")

In [ ]:
# --- Test 2.3: PromptProposalHead + decode_boxes proxy ---
print("Test 2.3 — PromptProposalHead forward pass...")

head = PromptProposalHead(embed_dim=256)
dummy_fused = torch.randn(2, 256, 64, 64)
box_off, logits = head(dummy_fused)
assert box_off.shape  == (2, 4, 64, 64), f"box_offsets shape: {box_off.shape}"
assert logits.shape   == (2, 1, 64, 64), f"logits shape: {logits.shape}"
print(f"  PASS  box_offsets={tuple(box_off.shape)}, logits={tuple(logits.shape)}")

In [ ]:
# --- Test 2.4: DSMLoss computation ---
print("Test 2.4 — DSMLoss computation...")

B, K = 2, 10
fake_outputs_dsm = {
    "pred_logits": torch.randn(B, K, 1),
    "pred_boxes":  torch.rand(B, K, 4) * 1024,
    "pred_masks":  torch.randn(B, K, 1, 256, 256),
}
fake_targets_dsm = [
    {"boxes": torch.rand(4, 4)*1024, "masks": torch.zeros(4, 1024, 1024), "labels": torch.ones(4, dtype=torch.long)},
    {"boxes": torch.rand(2, 4)*1024, "masks": torch.zeros(2, 1024, 1024), "labels": torch.ones(2, dtype=torch.long)},
]

matcher_dsm = DSMHungarianMatcher()
criterion_dsm = DSMLoss(matcher_dsm)
loss_dsm = criterion_dsm(fake_outputs_dsm, fake_targets_dsm)
assert loss_dsm.item() == loss_dsm.item(), "Loss is NaN!"
assert loss_dsm.item() > 0
print(f"  PASS  loss = {loss_dsm.item():.4f}  (scalar, > 0)")

In [ ]:
# --- Test 2.5: SAM2 config path resolution ---
print("Test 2.5 — SAM2 config file resolution...")

SAM2_CONFIG_REL = "configs/sam2.1/sam2.1_hiera_l.yaml"

# SAM2 resolves the config relative to its installed package directory
try:
    import sam2
    sam2_pkg_dir = os.path.dirname(sam2.__file__)
    # Hydra config is searched relative to the sam2 package's configs folder
    config_candidate = os.path.join(sam2_pkg_dir, "..", SAM2_CONFIG_REL)
    config_exists = os.path.exists(os.path.abspath(config_candidate))
    print(f"  SAM2 package dir: {sam2_pkg_dir}")
    print(f"  Config path: {os.path.abspath(config_candidate)}")
    if config_exists:
        print(f"  PASS  Config file found")
    else:
        # Check in current working directory too
        cwd_config = os.path.join(os.getcwd(), SAM2_CONFIG_REL)
        if os.path.exists(cwd_config):
            print(f"  PASS  Config found in CWD: {cwd_config}")
        else:
            print(f"  WARN  Config not found at either location.")
            print(f"         SAM2 may resolve it internally — test the full init to be sure.")
except ImportError:
    print("  SKIP  sam2 not installed")

In [ ]:
# --- Test 2.6: DSMPrompterSAM2 full forward pass (requires SAM2 checkpoint) ---
print("Test 2.6 — DSMPrompterSAM2 full forward pass...")

if not os.path.exists(SAM2_CHECKPOINT):
    print(f"  SKIP  SAM2 checkpoint not found: {SAM2_CHECKPOINT}")
    print("         Download sam2.1_hiera_large.pt and place it at:")
    print(f"         {SAM2_CHECKPOINT}")
    print("         Command: bash checkpoints/download_ckpts.sh  (from the SAM2 repo)")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"  Loading DSMPrompterSAM2 on {device} (this may take ~60s)...")

    try:
        model_dsm = DSMPrompterSAM2(SAM2_CONFIG_REL, SAM2_CHECKPOINT, num_proposals=5)
        model_dsm = model_dsm.to(device)
        model_dsm.eval()

        trainable = sum(p.numel() for p in model_dsm.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in model_dsm.parameters())
        print(f"  Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

        # Dummy inputs: RGB (B,3,1024,1024), DSM (B,1,1024,1024)
        dummy_rgb = torch.randn(1, 3, 1024, 1024, device=device)
        dummy_dsm = torch.randn(1, 1, 1024, 1024, device=device)

        with torch.no_grad():
            masks, ious, boxes, scores = model_dsm(dummy_rgb, dummy_dsm)

        assert masks.shape  == (1, 5, 1, 256, 256), f"masks shape: {masks.shape}"
        assert ious.shape   == (1, 5, 1),            f"ious shape: {ious.shape}"
        assert boxes.shape  == (1, 5, 4),            f"boxes shape: {boxes.shape}"
        assert scores.shape == (1, 5),               f"scores shape: {scores.shape}"
        print(f"  PASS  masks={tuple(masks.shape)}, ious={tuple(ious.shape)}, boxes={tuple(boxes.shape)}")

        # Test loss end-to-end
        targets_d = [{"boxes": torch.rand(3, 4, device=device)*1024,
                      "masks": torch.zeros(3, 1024, 1024, device=device),
                      "labels": torch.ones(3, dtype=torch.long, device=device)}]
        outputs_d = {"pred_logits": scores.unsqueeze(-1), "pred_boxes": boxes, "pred_masks": masks}
        loss_full = DSMLoss(DSMHungarianMatcher())(outputs_d, targets_d)
        print(f"  PASS  End-to-end loss = {loss_full.item():.4f}")

    except Exception as e:
        print(f"  FAIL  {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()

In [ ]:
# --- Test 2.7: Check decode_boxes overflow risk (DSMPrompter) ---
# RSPrompter clamps torch.exp() to [-10, 10] but DSMPrompter does not → NaN/Inf risk
print("Test 2.7 — DSMPrompter decode_boxes overflow check...")

# Simulate decode_boxes with extreme values
large_offset = torch.tensor([0.0, 0.0, 100.0, 100.0])  # exp(100) = Inf
clamped      = torch.exp(large_offset.clamp(-10, 10))   # safe
unclamped    = torch.exp(large_offset)                   # dangerous

has_overflow = torch.isinf(unclamped).any() or torch.isnan(unclamped).any()
clamped_safe = torch.isfinite(clamped).all()

print(f"  Without clamp: exp([0, 0, 100, 100]) = {unclamped.tolist()} → has inf/nan: {has_overflow}")
print(f"  With clamp:    exp(clamp([-10,10]))   = {clamped.tolist()} → all finite: {clamped_safe}")

if has_overflow:
    print()
    print("  WARN  DSMPrompter.decode_boxes does NOT clamp exp() offsets.")
    print("         This can cause NaN/Inf loss if the network produces large offsets early in training.")
    print("         RSPrompter uses .clamp(-10, 10) before exp() — consider applying the same fix.")
else:
    print("  OK  No overflow with normal offsets")

---
## 3. Dataset Check

Vérifie que les données COCO sont bien formatées et que les images sont accessibles.

In [ ]:
# --- Test 3.1: COCO JSON structure ---
print("Test 3.1 — COCO JSON structure...")

import json

for split, json_path in [("train", TRAIN_JSON), ("val", VAL_JSON), ("test", TEST_JSON)]:
    if not os.path.exists(json_path):
        print(f"  MISS  {split}: {json_path}")
        continue
    with open(json_path) as f:
        data = json.load(f)
    n_imgs  = len(data.get("images", []))
    n_anns  = len(data.get("annotations", []))
    n_cats  = len(data.get("categories", []))
    cat_ids = sorted([c["id"] for c in data.get("categories", [])])
    print(f"  OK    {split:5s}: {n_imgs:4d} images, {n_anns:5d} annotations, {n_cats} categories {cat_ids}")

In [ ]:
# --- Test 3.2: Image files accessibility ---
print("Test 3.2 — Image files accessible...")

for split in ["train", "val", "test"]:
    rgb_dir = os.path.join(DATA_ROOT, split, "RGB")
    dsm_dir = os.path.join(DATA_ROOT, split, "DSM")
    rgb_ok  = os.path.isdir(rgb_dir)
    dsm_ok  = os.path.isdir(dsm_dir)
    rgb_n   = len(os.listdir(rgb_dir)) if rgb_ok else 0
    dsm_n   = len(os.listdir(dsm_dir)) if dsm_ok else 0
    print(f"  {split:5s}: RGB {'OK' if rgb_ok else 'MISS'} ({rgb_n} files)  |  DSM {'OK' if dsm_ok else 'MISS'} ({dsm_n} files)")

---
## 4. Summary & Next Steps

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

checks = {
    "SAM ViT-H checkpoint":     os.path.exists(SAM_CHECKPOINT),
    "SAM2 checkpoint":           os.path.exists(SAM2_CHECKPOINT),
    "SAM embeddings (step 1)": os.path.isdir(os.path.join(EMBED_DIR, "train")),
    "train.json":                os.path.exists(TRAIN_JSON),
    "val.json":                  os.path.exists(VAL_JSON),
    "test.json":                 os.path.exists(TEST_JSON),
    "RGB train images":          os.path.isdir(os.path.join(DATA_ROOT, "train", "RGB")),
    "DSM train images":          os.path.isdir(os.path.join(DATA_ROOT, "train", "DSM")),
}

for label, ok in checks.items():
    print(f"  {'OK  ' if ok else 'MISS'}  {label}")

print()
print("NEXT STEPS:")
sam_ok  = checks["SAM ViT-H checkpoint"]
sam2_ok = checks["SAM2 checkpoint"]
emb_ok  = checks["SAM embeddings (step 1)"]

if not sam_ok:
    print("  1. Download SAM ViT-H: wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth")
if not sam2_ok:
    print("  2. Download SAM2: cd checkpoints && bash download_ckpts.sh  (or wget the .pt manually)")
if sam_ok and not emb_ok:
    print("  3. Pre-compute SAM embeddings: python src/misc/Pre_Compute_SAM.py")
if sam_ok and emb_ok:
    print("  3. Train RSPrompter: python src/models/RSPrompter.py")
if sam2_ok:
    print("  4. Train DSMPrompter: python src/models/DSM_Prompter.py")
    print("     (or via the dedicated script: python src/training/train_dsm_prompter.py)")